In [18]:
# Challenge 01 -- Supplier Performance Speed

# Statement:
# You are a supply chain analyst at a mid-size manufacturer. 
# Your procurement team has 3 years of purchase order data across 8 suppliers. 
# Leadership wants a quarterly supplier performance scorecard that ranks suppliers on four KPIs 
# and flags any supplier that is deteriorating over time.

# Challenge Data
import pandas as pd
import numpy as np
import plotly.express as px

np.random.seed(42)

suppliers = ["Acme", "GlobalCo", "FastParts", "PrimeMfg",
             "EastCoast", "WestSupply", "NorthTrade", "SouthCo"]

quarters = ["2022-Q1", "2022-Q2", "2022-Q3", "2022-Q4",
            "2023-Q1", "2023-Q2", "2023-Q3", "2023-Q4",
            "2024-Q1", "2024-Q2", "2024-Q3", "2024-Q4"]

rows = []
for supplier in suppliers:
    for quarter in quarters:
        rows.append({
            "supplier":        supplier,
            "quarter":         quarter,
            "po_count":        np.random.randint(10, 50),
            "on_time":         np.random.randint(7, 50),
            "total_delivered": np.random.randint(10, 50),
            "defects":         np.random.randint(0, 8),
            "spend":           round(np.random.uniform(10000, 80000), 2),
            "lead_time_days":  np.random.randint(5, 35)
        })

df = pd.DataFrame(rows)
df["on_time"] = df[["on_time", "total_delivered"]].min(axis=1)

print(df.shape)
print(df.head(10))

(96, 8)
  supplier  quarter  po_count  on_time  total_delivered  defects     spend  \
0     Acme  2022-Q1        48       24               24        2  64578.37   
1     Acme  2022-Q2        48       25               32        2  42147.42   
2     Acme  2022-Q3        45       33               33        2  11440.91   
3     Acme  2022-Q4        33       36               47        1  22727.75   
4     Acme  2023-Q1        42       18               31        4  40236.15   
5     Acme  2023-Q2        36       37               37        3  35645.33   
6     Acme  2023-Q3        12       16               16        4  41534.95   
7     Acme  2023-Q4        13       23               23        1  36979.16   
8     Acme  2024-Q1        11       26               37        6  57828.45   
9     Acme  2024-Q2        17       23               23        0  28114.60   

   lead_time_days  
0              25  
1              25  
2               6  
3              25  
4              21  
5            

In [19]:
# Part 1 -- Data Validation
# Check the data quality
# use f-strings as a software engineer would

# f-string notes:
# \nNewline — move to next line
# \tTab — indent horizontally
# \\Literal backslash
# \"Literal double quote inside a string
# common pattern: print(f"\n{section_title}:\n{data}")
# first \n — space before the section
# Second \n — separates the title from the data below it

# check the shape of the data
# df.shape returns a pair of two values -- a tuple (96, 8)
# it is not a method so it does not require parentheses
# df.shape[0] accesses the first element of the tuple -- zero-based - so returns the number of columns "96" in the (96,8) tuple
# df.shape[1] accesses the number of columns -- second value in the returned tuple "8"
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

# check the data for null values
# begin by defining null
# write f-strings
# us \n as a newline character -- telling python to move to the next line -- adding a blank between each f-string
nulls = df.isnull().sum()
print(f"\nNull values per column:\n{nulls}")
print(f"Total nulls: {nulls.sum()}")

# check for duplicate values
# define duplicate values
# write and f-string
dupes = df.duplicated().sum()
print(f"\nDuplicte rows: {dupes}")

# review the dataframes datatypes
print(f"\nColumn dtypes:\n{df.dtypes}")

# get descriptive statistics of the dataframe
print(f"\nDescriptive Statistics:\n {df.describe().round(2)}")

# review the business logic -- quick sanity check
issues = df[df["on_time"] > df["total_delivered"]]
print(f"\nBusiness Logic Violations (on_time > total_delivered): {len(issues)}")


Dataset shape: 96 rows x 8 columns

Null values per column:
supplier           0
quarter            0
po_count           0
on_time            0
total_delivered    0
defects            0
spend              0
lead_time_days     0
dtype: int64
Total nulls: 0

Duplicte rows: 0

Column dtypes:
supplier               str
quarter                str
po_count             int64
on_time              int64
total_delivered      int64
defects              int64
spend              float64
lead_time_days       int64
dtype: object

Descriptive Statistics:
        po_count  on_time  total_delivered  defects     spend  lead_time_days
count     96.00    96.00            96.00    96.00     96.00           96.00
mean      31.43    24.29            29.49     3.62  42175.95           19.79
std       11.83    10.51            11.62     2.44  19052.68            8.82
min       10.00     7.00            10.00     0.00  11265.28            5.00
25%       21.00    14.75            20.00     2.00  27830.12         

In [20]:
# Part 2 -- Calculate four KPI's per supplier per quarter
# otd_rate — on-time delivery rate: on_time / total_delivered * 100
# defect_rate — defect rate: defects / total_delivered * 100
# avg_lead_time — average lead time in days
# spend_share — each supplier's spend as a % of total spend that quarter

df = (
    df
    .assign(
        otd_rate = lambda x: (x["on_time"] / x["total_delivered"] * 100),
        defect_rate = lambda x: (x["defects"] / x["total_delivered"] * 100),
        avg_lead_time = lambda x: x.groupby(by=["supplier", "quarter"])["lead_time_days"].transform("mean"),
        total_spend_supplier = lambda x: x.groupby(by="supplier")["spend"].transform("sum"),
        spend_share = lambda x: (x["spend"] / x["total_spend_supplier"] * 100).round(2)
    )
)

print(df)

   supplier  quarter  po_count  on_time  total_delivered  defects     spend  \
0      Acme  2022-Q1        48       24               24        2  64578.37   
1      Acme  2022-Q2        48       25               32        2  42147.42   
2      Acme  2022-Q3        45       33               33        2  11440.91   
3      Acme  2022-Q4        33       36               47        1  22727.75   
4      Acme  2023-Q1        42       18               31        4  40236.15   
..      ...      ...       ...      ...              ...      ...       ...   
91  SouthCo  2023-Q4        45       18               28        3  34843.36   
92  SouthCo  2024-Q1        23       28               28        1  54228.60   
93  SouthCo  2024-Q2        14       21               21        0  48759.21   
94  SouthCo  2024-Q3        45       25               25        0  17942.52   
95  SouthCo  2024-Q4        48       20               40        1  40907.14   

    lead_time_days    otd_rate  defect_rate  avg_le

In [21]:
# Part 3 -- Build a Quarterly Scorecard
# For each supplier calculate their average across all 12 quarters for each KPI
# Produce a clean summary DataFrame showing all four KPI averages and all four ranks side by side.
scard = (
    df
    .groupby(
        by=["supplier"]
    )
    .agg(
        avg_otd_rate=("otd_rate", "mean"),
        avg_defect_rate=("defect_rate", "mean"),
        avg_lead_time_b=("lead_time_days", "mean"),
        avg_spend_share=("spend_share", "mean")
    )
    .round(2)
    .assign(
        rank_otd=lambda x: x["avg_otd_rate"].rank(ascending=False, method="dense").astype(int),
        rank_defect=lambda x: x["avg_defect_rate"].rank(ascending=True, method="dense").astype(int),
        rank_lead_time=lambda x: x["avg_lead_time_b"].rank(ascending=True, method="dense").astype(int),
        rank_spend_share=lambda x: x["avg_spend_share"].rank(ascending=False, method="dense").astype(int)
    )
    .sort_values("rank_otd")
    )
    
# view the averages and rankings side-by-side
print(scard[["avg_otd_rate","rank_otd", "avg_defect_rate", "rank_defect", "avg_lead_time_b", "rank_lead_time", "avg_spend_share", "rank_spend_share"]])

# notes
# FastParts has the highest ranked avg_otd_rate but also has an extremely high avg_defect_rate
# SouthCo has the lowest avg_defect_rate

            avg_otd_rate  rank_otd  avg_defect_rate  rank_defect  \
supplier                                                           
FastParts          96.44         1            22.45            7   
EastCoast          90.56         2            15.33            5   
NorthTrade         87.75         3            24.84            8   
GlobalCo           85.56         4            14.88            4   
Acme               83.75         5            10.94            3   
WestSupply         81.98         6            17.38            6   
SouthCo            80.26         7             7.63            1   
PrimeMfg           74.02         8            10.80            2   

            avg_lead_time_b  rank_lead_time  avg_spend_share  rank_spend_share  
supplier                                                                        
FastParts             22.33               5             8.33                 2  
EastCoast             17.33               3             8.33                

In [ ]:
# Part 4: Deteriorating Suppliers
# Deteriorating = ["otd_rate"].mean() in 2024 > 5 percentage points lower than their average otd_rate in 2022

# You need to compare each supplier's average otd_rate in 2022 vs 2024. 
# Think through:
# How do you isolate rows by year?
# How do you calculate a per-supplier average for each year?
# How do you compare two values from different filtered DataFrames?

 # step 1 — extract year and filter each year separately
df["year"] = df["quarter"].str[:4]

otd_2022 = (
    df[df["year"] == "2022"]
    .groupby("supplier")["otd_rate"]
    .mean()
    .rename("otd_2022")
)

otd_2024 = (
    df[df["year"] == "2024"]
    .groupby("supplier")["otd_rate"]
    .mean()
    .rename("otd_2024")
)

# step 2 — merge the two years together
comparison = pd.merge(otd_2022, otd_2024, on="supplier")

# step 3 — calculate the difference and filter
comparison = (
    comparison
    .assign(otd_change=lambda x: x["otd_2024"] - x["otd_2022"])
)

deteriorating = comparison[comparison["otd_change"] < -5]
print(f"Deteriorating suppliers:\n{deteriorating}")


AssertionError: 